In [ ]:
import json\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\nfrom src.models.t5_corrector import T5GrammarCorrector\nfrom src.utils.config import load_config\nfrom src.utils.evaluation import Evaluator\n\nconfig = load_config()\ndata_dir = Path(config.data.processed_data_path)

In [ ]:
def load_jsonl(path: Path):\n    with path.open('r', encoding='utf-8') as file_handle:\n        return [json.loads(line) for line in file_handle if line.strip()]\n\ntrain_path = data_dir / 'grammar_correction_train.jsonl'\nvalidation_path = data_dir / 'grammar_correction_validation.jsonl'\ntest_path = data_dir / 'grammar_correction_test.jsonl'\n\ntrain_rows = load_jsonl(train_path)\nvalidation_rows = load_jsonl(validation_path)\ntest_rows = load_jsonl(test_path)\nprint(len(train_rows), len(validation_rows), len(test_rows))

In [ ]:
corrector = T5GrammarCorrector(model_name=config.model.t5_model_name)\ncorrector.batch_size = config.model.batch_size\ncorrector.learning_rate = config.model.learning_rate\ncorrector.num_beams = config.model.num_beams\ncorrector.max_length = config.model.max_length\ncorrector

In [ ]:
try:\n    corrector.load_model()\n    print('Parameter count:', corrector._parameter_count())\n    print(corrector.model)\nexcept ImportError as exc:\n    print('Model dependencies not installed in this notebook environment:', exc)

In [ ]:
examples = [row['original'] for row in test_rows[:3]]\ntry:\n    baseline_predictions = [corrector.correct(text) for text in examples]\n    pd.DataFrame({'original': examples, 'prediction': baseline_predictions})\nexcept Exception as exc:\n    print('Pre-fine-tuning demo skipped:', exc)

In [ ]:
# Demo fine-tuning run (commented out for notebooks opened without model dependencies)\n# metrics = corrector.fine_tune(\n#     train_dataset=train_rows,\n#     val_dataset=validation_rows,\n#     output_dir='models/t5_demo',\n#     num_epochs=1,\n# )\n# metrics

In [ ]:
mock_losses = [2.15, 1.76, 1.42, 1.18]\nplt.figure(figsize=(8, 4))\nplt.plot(range(1, len(mock_losses) + 1), mock_losses, marker='o')\nplt.title('T5 Training Loss Curve (Demo)')\nplt.xlabel('Epoch')\nplt.ylabel('Loss')\nplt.grid(True, alpha=0.3)\nplt.show()

In [ ]:
evaluator = Evaluator()\ntry:\n    predictions = corrector.correct_batch([row['original'] for row in test_rows[:25]], batch_size=8)\n    references = [row['references'] for row in test_rows[:25]]\n    print('GLEU:', evaluator.compute_gleu(predictions, references))\n    print('ROUGE:', evaluator.compute_rouge(predictions, [row['corrected'] for row in test_rows[:25]]))\nexcept Exception as exc:\n    print('Evaluation skipped:', exc)

In [ ]:
failure_rows = []\ntry:\n    predictions = corrector.correct_batch([row['original'] for row in test_rows[:50]], batch_size=8)\n    for row, prediction in zip(test_rows[:50], predictions):\n        if prediction.strip() != row['corrected'].strip():\n            failure_rows.append({\n                'original': row['original'],\n                'prediction': prediction,\n                'reference': row['corrected'],\n            })\n    pd.DataFrame(failure_rows[:5])\nexcept Exception as exc:\n    print('Failure analysis skipped:', exc)

In [ ]:
try:\n    corrector.save('models/t5_notebook_checkpoint')\n    print('Saved checkpoint to models/t5_notebook_checkpoint')\nexcept Exception as exc:\n    print('Checkpoint save skipped:', exc)